# 12 · Layer 3 — Dual Storage (chunk vectors + claim-scoped graph)
Every node and edge carries a `claim_id`; the predicate schema is a whitelist of domain verbs and generic edges are rejected at insert time. Cross-claim network links live in a reserved scope reachable only through a separately-authorized API.

In [ ]:
# --- bootstrap: make the src package importable from any working dir ---
import sys
from pathlib import Path
p = Path.cwd().resolve()
while not (p / 'config' / '00_config.py').exists() and p != p.parent:
    p = p.parent
if str(p) not in sys.path:
    sys.path.insert(0, str(p))
print('project root:', p)


In [ ]:
from src.repository import Repository
from src import build_graph
from src.graph_store import get_graph_store, validate_predicate, PredicateRejected
repo = Repository()
stats = build_graph.build_graph(repo)
print({k: v for k, v in stats.items() if k != 'predicates'})
print('predicates:', stats['predicates'])


In [ ]:
# graph density control: generic predicates are rejected outright
for p in ('TREATED_BY', 'MENTIONED_IN', 'RELATED_TO'):
    try:
        print(f'  {p:14s} -> accepted as {validate_predicate(p)}')
    except PredicateRejected as e:
        print(f'  {p:14s} -> REJECTED: {str(e)[:70]}')


In [ ]:
g = get_graph_store(); g.load()
sub = g.subgraph('CLM0005')
print('CLM0005 subgraph:', len(sub['nodes']), 'nodes,', len(sub['edges']), 'edges')
for e in sub['edges'][:8]:
    print(f"   {e['subject'][:26]:28s} --{e['predicate']:22s}--> {e['object'][:24]:26s} "
          f"[{e['doc_id']}:{e['span'][0]}-{e['span'][1]}]")


In [ ]:
# cross-claim links exist but are NOT reachable from a claim scope
from src.graph_store import ScopeViolation, CROSS_CLAIM_SCOPE
try:
    g.neighbors([], 1, CROSS_CLAIM_SCOPE)
except ScopeViolation as e:
    print('blocked as designed:', str(e)[:90])
try:
    g.cross_claim_links(['x'], authorized=False)
except ScopeViolation as e:
    print('blocked as designed:', str(e)[:90])
repo.close()
